In [1]:
!pip install pandas numpy scikit-learn fairlearn pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.0/240.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 49.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 64.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 60.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
import os

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

In [3]:
def preprocess_adult_for_fair_bbn(csv_path='/kaggle/input/all-datasets/adult.csv'):
    df = pd.read_csv(csv_path)
    df = df[~df.isin(['?']).any(axis=1)].reset_index(drop=True)
    df['income'] = np.where(df['income']=='>50K', 1, 0)
    # Upsample positives
    more_50k = df[df['income']==1]
    dist = more_50k.groupby(['race','sex']).size().reset_index(name='count')
    dist['up_count'] = (dist['count']*22654/dist['count'].sum()).round().astype(int)
    ups = []
    for _, row in dist.iterrows():
        group = more_50k[(more_50k['race']==row['race']) & (more_50k['sex']==row['sex'])]
        ups.append(resample(group, replace=True, n_samples=row['up_count'], random_state=seed))
    df = pd.concat([df[df['income']==0]] + ups).sample(frac=1, random_state=seed).reset_index(drop=True)
    # Sensitive labels
    race_map = {"White": 0,"Black": 1,"Asian-Pac-Islander": 2,"Amer-Indian-Eskimo": 3,"Other": 4}
    sex_map = {"Male": 0,"Female": 1}
    df['race_label'] = df['race'].map(race_map)
    df['sex_label'] = df['sex'].map(sex_map)
    # Columns
    categorical_cols = ['workclass','education','marital.status','occupation','relationship','native.country']
    numeric_cols = ['age','fnlwgt','education.num','capital.gain','capital.loss','hours.per.week']
    # Preprocessor for NN
    preproc = ColumnTransformer([
        ('num', Pipeline([('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
    ])
    # BBN processing
    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = pd.qcut(bbn_df[col], 5, labels=False, duplicates='drop').astype(int)
    for col in categorical_cols + ['race','sex']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    # Split train/test
    X = df.drop(columns=['income','race_label','sex_label'])
    y = df['income'].values
    sens_race = df['race_label']
    sens_sex = df['sex_label']
    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_sex_train, sens_sex_test = \
        train_test_split(X, y, sens_race, sens_sex, test_size=0.2, stratify=y, random_state=seed)
    # NN preprocessing
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    # BBN datasets aligned
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    return {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_sex_train': sens_sex_train.reset_index(drop=True),
        'sens_sex_test': sens_sex_test.reset_index(drop=True)
    }

In [4]:
def preprocess_compas_for_fair_bbn(csv_path='/kaggle/input/all-datasets/compas-scores-two-years.csv', seed=42):
    # Load and filter COMPAS data
    df = pd.read_csv(csv_path)
    df = df.loc[
        (df['days_b_screening_arrest'] <= 30) &
        (df['days_b_screening_arrest'] >= -30) &
        (df['is_recid'] != -1) &
        (df['c_charge_degree'] != 'O') &
        (df['score_text'] != 'N/A'),
        ['age','c_charge_degree','race','age_cat','score_text','sex',
         'priors_count','days_b_screening_arrest','decile_score',
         'juv_other_count','juv_misd_count','juv_fel_count',
         'c_charge_desc','is_recid','two_year_recid','c_jail_in','c_jail_out']
    ].reset_index(drop=True)
    # Sensitive label mapping
    race_map = {"African-American":0,"Caucasian":1,"Hispanic":2,"Other":3,"Asian":4,"Native American":5}
    sex_map = {"Male":0,"Female":1}
    df['race_label'] = df['race'].map(race_map)
    df['sex_label'] = df['sex'].map(sex_map)
    # Jail time calculation
    df['c_jail_in'] = pd.to_datetime(df['c_jail_in'])
    df['c_jail_out'] = pd.to_datetime(df['c_jail_out'])
    df['jail_time'] = (df['c_jail_out'] - df['c_jail_in']).dt.days
    df['jail_time'] = df['jail_time'].fillna(0)
    df = df.drop(columns=['c_jail_in','c_jail_out'])
    # Target variable
    y = df['two_year_recid'].values
    sens_race = df['race_label']
    sens_sex = df['sex_label']
    # Columns
    numeric_cols = ['age','priors_count','days_b_screening_arrest','decile_score',
                    'jail_time','juv_other_count','juv_misd_count','juv_fel_count']
    categorical_cols = ['c_charge_degree','race','age_cat','score_text','sex','c_charge_desc']
    # Preprocessor for NN (scaled + one-hot)
    preproc = ColumnTransformer([
        ('num', Pipeline([('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
    ])
    # BBN preprocessing: discretize numerics, encode categoricals
    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = pd.qcut(bbn_df[col], 5, labels=False, duplicates='drop').astype(int)
    for col in categorical_cols + ['race','sex']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    # Split train/test
    X = df.drop(columns=['is_recid','two_year_recid','race_label','sex_label'])
    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_sex_train, sens_sex_test = \
        train_test_split(X, y, sens_race, sens_sex, test_size=0.2, stratify=y, random_state=seed)
    # NN preprocessing
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    # BBN aligned splits
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    return {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_sex_train': sens_sex_train.reset_index(drop=True),
        'sens_sex_test': sens_sex_test.reset_index(drop=True)
    }

In [5]:
def preprocess_german_for_fair_bbn(csv_path="/kaggle/input/all-datasets/german.data", seed=42):
    column_names = [
        "status", "duration", "credit_history", "purpose", "amount", "savings", "employment",
        "installment_rate", "personal_status_sex", "other_debtors", "residence", "property", "age",
        "other_installment_plans", "housing", "number_credits", "job", "people_liable",
        "telephone", "foreign_worker", "credit"
    ]
    df = pd.read_csv(csv_path, sep=' ', names=column_names)
    sex_map = {'A91':'male','A92':'male','A93':'male','A94':'female','A95':'female'}
    df['sex'] = df['personal_status_sex'].map(sex_map)
    df['sex_label'] = df['sex'].map({'male':0,'female':1})
    df['age_label'] = (df['age'] >= 25).astype(int)
    df['foreign_worker_label'] = df['foreign_worker'].map({'A201':1,'A202':0})
    df['credit_label'] = df['credit'].map({1:1,2:0})
    df = df.drop(columns=['personal_status_sex','sex','age','foreign_worker','credit'])
    def upsample_minority_groups_german(data, target_col, group_cols, target_value, total_target_samples):
        minority = data[data[target_col]==target_value]
        dist = minority.groupby(group_cols).size().reset_index(name='count')
        dist['up_count'] = (dist['count']*total_target_samples/dist['count'].sum()).round().astype(int)
        ups = []
        for _, row in dist.iterrows():
            group = minority
            for col in group_cols:
                group = group[group[col]==row[col]]
            ups.append(resample(group, replace=True, n_samples=row['up_count'], random_state=seed))
        return pd.concat([data[data[target_col]!=target_value]] + ups).sample(frac=1, random_state=seed).reset_index(drop=True)
    df = upsample_minority_groups_german(
        data=df,
        target_col='credit_label',
        group_cols=['sex_label','age_label','foreign_worker_label'],
        target_value=0,
        total_target_samples=700
    )
    X = df.drop(columns=['credit_label'])
    y = df['credit_label'].values
    sensitive_sex = df['sex_label'].values
    sensitive_age = df['age_label'].values
    sensitive_foreign = df['foreign_worker_label'].values
    numerical_cols = ["duration", "amount", "installment_rate", "residence","number_credits","people_liable"]
    categorical_cols = [col for col in X.columns if col not in numerical_cols]
    preproc = ColumnTransformer([
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])
    bbn_df = df.copy()
    for col in numerical_cols:
        bbn_df[col] = pd.qcut(bbn_df[col], 5, labels=False, duplicates='drop').astype(int)
    for col in categorical_cols + ['sex_label','age_label','foreign_worker_label']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    X_train_raw, X_test_raw, y_train, y_test, sens_age_train, sens_age_test, sens_sex_train, sens_sex_test, sens_foreign_train, sens_foreign_test = train_test_split(
        X, y, sensitive_age, sensitive_sex, sensitive_foreign,
        test_size=0.2, random_state=seed, stratify=y
    )
    pipeline = Pipeline([('preprocessor', preproc)])
    X_train_nn = pipeline.fit_transform(X_train_raw)
    X_test_nn = pipeline.transform(X_test_raw)
    bbn_train_df = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test_df = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    return {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train_df, 'bbn_test_df': bbn_test_df,
        'y_train': y_train, 'y_test': y_test,
        'sens_age_train': sens_age_train, 'sens_age_test': sens_age_test,
        'sens_sex_train': sens_sex_train, 'sens_sex_test': sens_sex_test,
        'sens_foreign_train': sens_foreign_train, 'sens_foreign_test': sens_foreign_test
    }

In [6]:
def preprocess_lawschool_for_fair_bbn(law_path="/kaggle/input/all-datasets/bar_pass_prediction.csv"):
    law_df = pd.read_csv(law_path)
    law_df.columns = [c.strip().lower() for c in law_df.columns]
    target_col, sens_race, sens_gender = 'pass_bar', 'race', 'sex'
    # Drop rows missing critical columns
    law_df = law_df.dropna(subset=[target_col, sens_race, sens_gender])
    # Fill numeric NaNs with median
    for col in law_df.select_dtypes(include=[np.number]).columns:
        law_df[col] = law_df[col].fillna(law_df[col].median())
    # Clip numeric outliers
    for col in law_df.select_dtypes(include=[np.number]).columns:
        q1, q3 = law_df[col].quantile(0.25), law_df[col].quantile(0.75)
        iqr = q3 - q1
        if iqr > 0:
            law_df[col] = np.clip(law_df[col], q1 - 1.5*iqr, q3 + 1.5*iqr)
    # Binary income label
    law_df['income'] = np.where(law_df['fam_inc'] > law_df['fam_inc'].median(), 1, 0)
    # Upsample high income group
    high_income = law_df[law_df['income'] == 1]
    dist = high_income.groupby([sens_race, sens_gender]).size().reset_index(name='count')
    target_count = len(law_df[law_df['income'] == 0])
    dist['up_count'] = (dist['count'] * target_count / dist['count'].sum()).round().astype(int)
    ups = []
    for _, row in dist.iterrows():
        group = high_income[(high_income[sens_race] == row[sens_race]) & (high_income[sens_gender] == row[sens_gender])]
        if len(group) > 0:
            ups.append(resample(group, replace=True, n_samples=int(row['up_count']), random_state=seed))
    if ups:
        law_df = pd.concat([law_df[law_df['income'] == 0]] + ups).sample(frac=1, random_state=seed).reset_index(drop=True)
    # Encode sensitive attributes
    law_df[sens_race] = law_df[sens_race].astype('category')
    law_df[sens_gender] = law_df[sens_gender].astype('category')
    law_df['race_label'] = law_df[sens_race].cat.codes
    law_df['gender_label'] = law_df[sens_gender].cat.codes
    # Column groups
    numeric_cols = [c for c in ['lsat','ugpa','zfygpa','zgpa','age','gpa','fam_inc'] if c in law_df.columns]
    categorical_cols = [c for c in ['tier','cluster','fulltime','parttime'] if c in law_df.columns]
    # Drop low variance cols
    low_var_cols = [col for col in numeric_cols if law_df[col].nunique() <= 1]
    if low_var_cols:
        print(f"Dropping low-variance columns: {low_var_cols}")
        law_df = law_df.drop(columns=low_var_cols)
        numeric_cols = [c for c in numeric_cols if c not in low_var_cols]
    # Preprocessor
    preproc = ColumnTransformer([
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])
    # Discretized copy for BBN
    bbn_df = law_df.copy()
    for col in numeric_cols:
        if law_df[col].nunique() > 1:
            try:
                bbn_df[col] = pd.qcut(law_df[col], 5, labels=False, duplicates='drop')
            except:
                bbn_df[col] = pd.cut(law_df[col], 5, labels=False, duplicates='drop')
        else:
            bbn_df[col] = 0
        bbn_df[col] = bbn_df[col].fillna(0).astype(int)
    for col in categorical_cols + [sens_race, sens_gender]:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    X = law_df[numeric_cols + categorical_cols + [sens_race, sens_gender]]
    y = law_df['income'].values
    sens_race_labels = law_df['race_label']
    sens_gender_labels = law_df['gender_label']
    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_gender_train, sens_gender_test = \
        train_test_split(X, y, sens_race_labels, sens_gender_labels, test_size=0.2, stratify=y, random_state=seed)
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    print(f"Final dataset — X_train_nn: {X_train_nn.shape}, BBN_train: {bbn_train.shape}")
    return {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_gender_train': sens_gender_train.reset_index(drop=True),
        'sens_gender_test': sens_gender_test.reset_index(drop=True)
    }

In [7]:
def preprocess_diabetes_hospital_for_fair_bbn(csv_path='/kaggle/input/all-datasets/diabetes_hospital_fairlearn.csv', seed=42):
    """
    Preprocess diabetes hospital readmission dataset for fair BBN analysis.
    Similar structure to other dataset preprocessors.
    """
    df = pd.read_csv(csv_path)

    # Drop unwanted columns if they exist
    drop_cols = ["max_glu_serum", "A1Cresult", "readmitted.1"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])

    # Remove rows containing 'Missing' or NaN values
    df = df[~df.isin(['Missing']).any(axis=1)]
    df = df.dropna(subset=['race', 'gender']).reset_index(drop=True)

    # Convert age ranges to numerical midpoint or lower bound
    age_map = {
        "'0-10'": 5, "'10-20'": 15, "'20-30'": 25, "'30-40'": 35, "'40-50'": 45,
        "'50-60'": 55, "'60-70'": 65, "'70-80'": 75, "'80-90'": 85, "'90-100'": 95,
        "'30 years or younger'": 20,
        "'30-60 years'": 45,
        "'Over 60 years'": 70
    }
    df['age'] = df['age'].replace(age_map).astype(float)

    # Ensure target is numeric
    df['readmit_binary'] = df['readmit_binary'].astype(int)

    # Clean and encode sensitive attributes
    df['race'] = df['race'].astype(str).str.strip()
    df['gender'] = df['gender'].astype(str).str.strip()

    race_map = {v: i for i, v in enumerate(df['race'].unique())}
    gender_map = {'Male': 0, 'Female': 1}

    df['race_label'] = df['race'].map(race_map)
    df['gender_label'] = df['gender'].map(gender_map)

    # Fill unmapped or missing sensitive values safely
    df['race_label'] = df['race_label'].fillna(0).astype(int)
    df['gender_label'] = df['gender_label'].fillna(0).astype(int)

    # Define categorical and numeric feature columns
    categorical_cols = [
        'discharge_disposition_id', 'admission_source_id',
        'medical_specialty', 'primary_diagnosis',
        'insulin', 'change', 'diabetesMed'
    ]
    numeric_cols = [
        'age', 'time_in_hospital', 'num_lab_procedures', 'num_procedures',
        'num_medications', 'number_diagnoses', 'had_emergency',
        'had_inpatient_days', 'had_outpatient_days', 'medicare', 'medicaid'
    ]

    # Column transformer for NN preprocessing
    preproc = ColumnTransformer([
        ('num', Pipeline([('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
    ])

    # BBN processing: discretize numeric + encode categorical
    bbn_df = df.copy()
    for col in numeric_cols:
        if pd.api.types.is_numeric_dtype(bbn_df[col]):
            bbn_df[col] = pd.qcut(bbn_df[col], 5, labels=False, duplicates='drop').astype(int)
        else:
            print(f"⚠️ Warning: Column '{col}' is not numeric and will not be discretized.")

    for col in categorical_cols + ['race', 'gender']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)

    # Train-test split
    X = df.drop(columns=['readmit_binary', 'race_label', 'gender_label'])
    y = df['readmit_binary'].values
    sens_race = df['race_label']
    sens_gender = df['gender_label']

    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_gender_train, sens_gender_test = \
        train_test_split(X, y, sens_race, sens_gender, test_size=0.2, stratify=y, random_state=seed)

    # NN preprocessing
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)

    # Align BBN datasets
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)

    # Return everything neatly
    return {
        'X_train_nn': X_train_nn,
        'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train,
        'bbn_test_df': bbn_test,
        'y_train': y_train,
        'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_sex_train': sens_gender_train.reset_index(drop=True),   # keeping key name consistent
        'sens_sex_test': sens_gender_test.reset_index(drop=True)
    }


In [8]:
def preprocess_bank_for_fair_bbn(csv_path='/kaggle/input/all-datasets/bank-full.csv', seed=42):
    import pandas as pd
    import numpy as np
    from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
    from sklearn.compose import ColumnTransformer
    from sklearn.model_selection import train_test_split
    from sklearn.utils import resample
    try:
        df = pd.read_csv(csv_path, sep=';')
    except:
        df = pd.read_csv(csv_path, sep=',')
    df = df.apply(lambda x: x.str.lower() if x.dtype == 'object' else x)
    target_col = 'y' if 'y' in df.columns else 'deposit' if 'deposit' in df.columns else 'subscribed'
    if target_col not in df.columns:
        target_col = df.columns[-1]
    df = df[~df.isin(['unknown']).any(axis=1)].reset_index(drop=True)
    df['y'] = np.where(df[target_col].isin(['yes', 'y', 'true', '1']), 1, 0)
    positives = df[df['y'] == 1]
    negatives = df[df['y'] == 0]
    if len(positives) == 0:
        df.loc[df.head(100).index, 'y'] = 1
        positives = df[df['y'] == 1]
        negatives = df[df['y'] == 0]
    dist = positives.groupby(['job', 'marital']).size().reset_index(name='count')
    dist['up_count'] = (dist['count'] * len(negatives) / dist['count'].sum()).round().astype(int)
    upsamples = []
    for _, row in dist.iterrows():
        group = positives[(positives['job'] == row['job']) & (positives['marital'] == row['marital'])]
        if len(group) > 0:
            upsamples.append(resample(group, replace=True, n_samples=row['up_count'], random_state=seed))
    df = pd.concat([negatives] + upsamples).sample(frac=1, random_state=seed).reset_index(drop=True)
    le_marital = LabelEncoder()
    df['marital_label'] = le_marital.fit_transform(df['marital'])
    le_job = LabelEncoder()
    df['job_label'] = le_job.fit_transform(df['job'])
    categorical_cols = [col for col in ['job', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome'] if col in df.columns]
    numeric_cols = [col for col in ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'] if col in df.columns]
    for col in ['balance', 'duration', 'pdays', 'previous']:
        if col in df.columns:
            df[col] = df[col].clip(upper=df[col].quantile(0.99))
    preproc = ColumnTransformer([
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])
    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = pd.qcut(bbn_df[col], 5, labels=False, duplicates='drop').astype(int)
    for col in categorical_cols + ['marital', 'job']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    X = df.drop(columns=['y', 'marital_label', 'job_label'])
    y = df['y'].values
    sens_marital = df['marital_label']
    sens_job = df['job_label']
    X_train_raw, X_test_raw, y_train, y_test, sens_marital_train, sens_marital_test, sens_job_train, sens_job_test = \
        train_test_split(X, y, sens_marital, sens_job, test_size=0.2, stratify=y, random_state=seed)
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    return {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_marital_train': sens_marital_train.reset_index(drop=True),
        'sens_marital_test': sens_marital_test.reset_index(drop=True),
        'sens_job_train': sens_job_train.reset_index(drop=True),
        'sens_job_test': sens_job_test.reset_index(drop=True)
    }

In [9]:
class SimpleFairBBN:
    def __init__(self, feature_names=None, target_name='y'):
        self.feature_names = feature_names or []
        self.target_name = target_name
        self.model = None
        self.inference = None

    def fit(self, df_discrete, y, include_sensitive=True):
        data = df_discrete.copy().reset_index(drop=True)
        data[self.target_name] = y
        candidate_features = list(df_discrete.columns)
        selected = candidate_features[:6]
        if include_sensitive:
            for sens in ['marital', 'job', 'race', 'sex']:
                if sens in df_discrete.columns and sens not in selected:
                    selected.append(sens)
        edges = [(f, self.target_name) for f in selected]
        self.feature_names = selected
        self.model = DiscreteBayesianNetwork(edges)
        self.model.fit(data, estimator=BayesianEstimator, prior_type='BDeu', equivalent_sample_size=5)
        self.inference = VariableElimination(self.model)

    def predict_proba(self, df_discrete):
        Xdf = df_discrete.reset_index(drop=True)
        probs = []
        for _, row in Xdf.iterrows():
            evidence = {}
            used = 0
            for f in self.feature_names:
                if f in row and not pd.isna(row[f]) and used < 3:
                    try:
                        evidence[f] = int(row[f])
                        used += 1
                    except:
                        continue
            try:
                q = self.inference.query(variables=[self.target_name], evidence=evidence) if evidence else self.inference.query(variables=[self.target_name])
                p1 = q.values[1] if len(q.values) == 2 else 0.5
            except:
                p1 = 0.5
            probs.append(p1)
        return np.array(probs)

In [10]:
class AdapterNN(nn.Module):
    def __init__(self, in_dim=3, hidden_dim=32):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.out = nn.Linear(hidden_dim // 2, 1)

    def forward(self, x, return_features=False):
        h = self.act(self.fc1(x))
        h2 = self.act(self.fc2(h))
        logit = self.out(h2)
        return (logit, h2) if return_features else logit

In [11]:
class AdversaryNN(nn.Module):
    def __init__(self, in_dim, marital_classes, job_classes):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU())
        self.marital_head = nn.Linear(32, marital_classes)
        self.job_head = nn.Linear(32, job_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.marital_head(s), self.job_head(s)

class AdversaryNN_MEPS(nn.Module):
    def __init__(self, in_dim, race_classes, sex_classes):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU())
        self.race_head = nn.Linear(32, race_classes)
        self.sex_head = nn.Linear(32, sex_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.race_head(s), self.sex_head(s)

class AdversaryNN_LawSchool(nn.Module):
    def __init__(self, in_dim, race_classes, gender_classes):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU())
        self.race_head = nn.Linear(32, race_classes)
        self.gender_head = nn.Linear(32, gender_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.race_head(s), self.gender_head(s)

class AdversaryNN_Adult(nn.Module):
    def __init__(self, in_dim, race_classes, sex_classes):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU())
        self.race_head = nn.Linear(32, race_classes)
        self.sex_head = nn.Linear(32, sex_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.race_head(s), self.sex_head(s)

class AdversaryNN_COMPAS(nn.Module):
    def __init__(self, in_dim, race_classes, sex_classes):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU())
        self.race_head = nn.Linear(32, race_classes)
        self.sex_head = nn.Linear(32, sex_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.race_head(s), self.sex_head(s)

class AdversaryNN_German(nn.Module):
    def __init__(self, in_dim, age_classes, sex_classes, foreign_classes):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(in_dim, 32),
            nn.ReLU()
        )
        self.age_head = nn.Linear(32, age_classes)
        self.sex_head = nn.Linear(32, sex_classes)
        self.foreign_head = nn.Linear(32, foreign_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.age_head(s), self.sex_head(s), self.foreign_head(s)

In [12]:
class AdversaryNN_Hospital(nn.Module):
    def __init__(self, in_dim, race_classes, sex_classes):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU())
        self.race_head = nn.Linear(32, race_classes)
        self.sex_head = nn.Linear(32, sex_classes)

    def forward(self, x):
        s = self.shared(x)
        return self.race_head(s), self.sex_head(s)


In [13]:
def train_fair_bbn_adapter(data_dict, dataset_type, lambda_adv=0.5, alpha_bbn=0.5, epochs=60, batch_size=256, lr=1e-3):
    X_train_nn, X_test_nn = data_dict['X_train_nn'], data_dict['X_test_nn']
    bbn_train_df, bbn_test_df = data_dict['bbn_train_df'], data_dict['bbn_test_df']
    y_train, y_test = data_dict['y_train'], data_dict['y_test']

    config = {
        'german': {
            'sens_attrs': [('sens_age_train', 'sens_age_test'), ('sens_sex_train', 'sens_sex_test'), ('sens_foreign_train', 'sens_foreign_test')],
            'marginal_features': [['age_label'], ['sex_label'], ['foreign_worker_label']],
            'adversary_class': 'AdversaryNN_German',
            'adversary_kwargs': {'age_classes': 2, 'sex_classes': 2, 'foreign_classes': 2},
            'target_name': None,
        },
        'compas': {
            'sens_attrs': [('sens_race_train', 'sens_race_test'), ('sens_sex_train', 'sens_sex_test')],
            'marginal_features': [['race'], ['sex']],
            'adversary_class': 'AdversaryNN_COMPAS',
            'target_name': 'two_year_recid',
        },
        'adult': {
            'sens_attrs': [('sens_race_train', 'sens_race_test'), ('sens_sex_train', 'sens_sex_test')],
            'marginal_features': [['race'], ['sex']],
            'adversary_class': 'AdversaryNN_Adult',
            'target_name': None,
        },
        'bank': {
            'sens_attrs': [('sens_marital_train', 'sens_marital_test'), ('sens_job_train', 'sens_job_test')],
            'marginal_features': [['marital'], ['job']],
            'adversary_class': 'AdversaryNN',
            'features_subset': ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous', 'marital', 'job'],
            'target_name': 'y',
        },
        'lawschool': {
            'sens_attrs': [('sens_race_train', 'sens_race_test'), ('sens_gender_train', 'sens_gender_test')],
            'marginal_features': [['race'], ['gender']],
            'adversary_class': 'AdversaryNN_LawSchool',
            'features_subset': ['lsat', 'ugpa', 'fulltime', 'fam_inc', 'age', 'race', 'gender'],
            'target_name': 'pass_bar',
        },
        'hospital': {
            'sens_attrs': [('sens_race_train', 'sens_race_test'), ('sens_sex_train', 'sens_sex_test')],
            'marginal_features': [['race'], ['gender']],
            'adversary_class': 'AdversaryNN_Hospital',
            'target_name': 'target',
        },
    }

    cfg = config[dataset_type]
    
    if 'features_subset' in cfg:
        features = [f for f in cfg['features_subset'] if f in bbn_train_df.columns]
        bbn_train_sub = bbn_train_df[features]
        bbn_test_sub = bbn_test_df[features]
    else:
        bbn_train_sub = bbn_train_df
        bbn_test_sub = bbn_test_df

    if cfg.get('target_name') is None:
        cfg['target_name'] = 'target'

    print("Training baseline MLP...")
    baseline = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=seed)
    baseline.fit(X_train_nn, y_train)
    base_pred = baseline.predict(X_test_nn)
    base_acc = accuracy_score(y_test, base_pred)

    base_results = {'pred': base_pred, 'acc': base_acc}
    for i, (train_attr, test_attr) in enumerate(cfg['sens_attrs']):
        sens_train = data_dict[train_attr]
        sens_test = data_dict[test_attr]
        attr_name = train_attr.replace('sens_', '').replace('_train', '')
        base_results[f'{attr_name}_dp'] = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_test)
        base_results[f'{attr_name}_eo'] = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_test)

    print(f"Baseline MLP Accuracy: {base_acc:.4f}")

    print("Training fair BBN...")
    bbn = SimpleFairBBN(feature_names=list(bbn_train_sub.columns), target_name=cfg.get('target_name'))
    bbn.fit(bbn_train_sub, y_train, include_sensitive=True)

    p_all = bbn.predict_proba(bbn_train_sub).reshape(-1, 1)
    p_all_test = bbn.predict_proba(bbn_test_sub).reshape(-1, 1)
    
    marginals_train = [p_all]
    marginals_test = [p_all_test]
    
    for feat in cfg['marginal_features']:
        marginals_train.append(bbn.predict_proba(bbn_train_sub[feat]).reshape(-1, 1))
        marginals_test.append(bbn.predict_proba(bbn_test_sub[feat]).reshape(-1, 1))
    
    adapter_in_train = torch.tensor(np.hstack(marginals_train), dtype=torch.float32)
    adapter_in_test = torch.tensor(np.hstack(marginals_test), dtype=torch.float32)

    y_train_t = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
    y_test_t = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32)

    sens_tensors = []
    for train_attr, test_attr in cfg['sens_attrs']:
        sens_train = data_dict[train_attr]
        if hasattr(sens_train, 'values'):
            sens_train = sens_train.values
        sens_tensors.append(torch.tensor(sens_train.astype(int), dtype=torch.long))

    train_loader = DataLoader(
        TensorDataset(adapter_in_train, y_train_t, *sens_tensors),
        batch_size=batch_size, shuffle=True
    )

    adapter = AdapterNN(in_dim=len(marginals_train), hidden_dim=64)
    
    adv_kwargs = {'in_dim': 32}
    if dataset_type == 'german':
        adv_kwargs.update({'age_classes': 2, 'sex_classes': 2, 'foreign_classes': 2})
        adversary = AdversaryNN_German(**adv_kwargs)
    elif dataset_type == 'compas':
        adv_kwargs.update({'race_classes': len(data_dict['sens_race_train'].unique()), 'sex_classes': len(data_dict['sens_sex_train'].unique())})
        adversary = AdversaryNN_COMPAS(**adv_kwargs)
    elif dataset_type == 'adult':
        adv_kwargs.update({'race_classes': len(data_dict['sens_race_train'].unique()), 'sex_classes': len(data_dict['sens_sex_train'].unique())})
        adversary = AdversaryNN_Adult(**adv_kwargs)
    elif dataset_type == 'bank':
        adv_kwargs.update({'marital_classes': len(data_dict['sens_marital_train'].unique()), 'job_classes': len(data_dict['sens_job_train'].unique())})
        adversary = AdversaryNN(**adv_kwargs)
    elif dataset_type == 'lawschool':
        adv_kwargs.update({'race_classes': len(data_dict['sens_race_train'].unique()), 'gender_classes': len(data_dict['sens_gender_train'].unique())})
        adversary = AdversaryNN_LawSchool(**adv_kwargs)
    elif dataset_type == 'hospital':
        adv_kwargs.update({'race_classes': len(data_dict['sens_race_train'].unique()), 'sex_classes': len(data_dict['sens_sex_train'].unique())})
        adversary = AdversaryNN_Hospital(**adv_kwargs)

    adapter_opt = optim.Adam(adapter.parameters(), lr=lr)
    adv_opt = optim.Adam(adversary.parameters(), lr=lr)
    pred_loss_fn = nn.BCEWithLogitsLoss()
    adv_loss_fn = nn.CrossEntropyLoss()

    print("Training adapter with adversarial + BBN fairness...")
    for epoch in range(epochs):
        adapter.train(); adversary.train()
        total_adapter_loss = 0.0; total_adv_loss = 0.0

        for batch in train_loader:
            batch_in, batch_y = batch[0], batch[1]
            batch_sens = batch[2:]

            with torch.no_grad():
                _, features = adapter(batch_in, return_features=True)
            adv_opt.zero_grad()
            
            if dataset_type == 'german':
                age_logits, sex_logits, foreign_logits = adversary(features)
                adv_loss = (adv_loss_fn(age_logits, batch_sens[0]) + adv_loss_fn(sex_logits, batch_sens[1]) + adv_loss_fn(foreign_logits, batch_sens[2])) / 3.0
            elif dataset_type == 'bank':
                m_logits, j_logits = adversary(features)
                adv_loss = (adv_loss_fn(m_logits, batch_sens[0]) + adv_loss_fn(j_logits, batch_sens[1])) / 2.0
            else:
                logits = adversary(features)
                adv_loss = sum(adv_loss_fn(logits[i], batch_sens[i]) for i in range(len(batch_sens))) / len(batch_sens)

            adv_loss.backward(); adv_opt.step(); total_adv_loss += adv_loss.item()

            adapter_opt.zero_grad()
            logit, features = adapter(batch_in, return_features=True)
            pred_loss = pred_loss_fn(logit, batch_y)

            if dataset_type == 'german':
                age_logits2, sex_logits2, foreign_logits2 = adversary(features)
                adv_loss_for_adapter = (adv_loss_fn(age_logits2, batch_sens[0]) + adv_loss_fn(sex_logits2, batch_sens[1]) + adv_loss_fn(foreign_logits2, batch_sens[2])) / 3.0
                dp_penalty = torch.abs(features[batch_sens[1]==0].mean() - features[batch_sens[1]==1].mean())
            elif dataset_type == 'bank':
                m_logits2, j_logits2 = adversary(features)
                adv_loss_for_adapter = (adv_loss_fn(m_logits2, batch_sens[0]) + adv_loss_fn(j_logits2, batch_sens[1])) / 2.0
                grp0_mean = features[batch_sens[0] == 0].mean(dim=0)
                grp1_mean = features[batch_sens[0] == 1].mean(dim=0)
                dp_penalty = torch.abs(grp0_mean - grp1_mean).mean()
            else:
                logits2 = adversary(features)
                adv_loss_for_adapter = sum(adv_loss_fn(logits2[i], batch_sens[i]) for i in range(len(batch_sens))) / len(batch_sens)
                grp0_mean = features[batch_sens[0] == 0].mean(dim=0)
                grp1_mean = features[batch_sens[0] == 1].mean(dim=0)
                dp_penalty = torch.abs(grp0_mean - grp1_mean).mean()

            total_loss = pred_loss - lambda_adv * adv_loss_for_adapter + alpha_bbn * dp_penalty
            total_loss.backward(); adapter_opt.step()
            total_adapter_loss += total_loss.item()

        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch:3d} | Adv Loss: {total_adv_loss/len(train_loader):.4f} | Adapter Loss: {total_adapter_loss/len(train_loader):.4f}")

    adapter.eval(); adversary.eval()
    with torch.no_grad():
        test_logit, _ = adapter(adapter_in_test, return_features=True)
        test_probs = torch.sigmoid(test_logit.cpu()).numpy().flatten()
        test_pred = (test_probs > 0.5).astype(int)

    adv_acc = accuracy_score(y_test, test_pred)
    adv_results = {'pred': test_pred, 'acc': adv_acc}
    
    for i, (train_attr, test_attr) in enumerate(cfg['sens_attrs']):
        sens_test = data_dict[test_attr]
        attr_name = train_attr.replace('sens_', '').replace('_train', '')
        adv_results[f'{attr_name}_dp'] = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_test)
        adv_results[f'{attr_name}_eo'] = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_test)

    print("\nBASELINE MLP RESULTS:")
    print(f" Accuracy: {base_acc:.4f}")
    for key, val in base_results.items():
        if key not in ['pred', 'acc']:
            print(f" {key}: {val:.4f}")

    print("\nBBN + Adapter RESULTS:")
    print(f" Accuracy: {adv_acc:.4f}")
    for key, val in adv_results.items():
        if key not in ['pred', 'acc']:
            print(f" {key}: {val:.4f}")

    return {'baseline': base_results, 'bbn_adapter': adv_results}

In [14]:
if __name__ == '__main__':
    print("Starting Fair BBN System on Kaggle")
    print("=" * 60)

    datasets = [
        ("GERMAN CREDIT", preprocess_german_for_fair_bbn, "german"),
        ("COMPAS", preprocess_compas_for_fair_bbn, "compas"),
        ("ADULT INCOME", preprocess_adult_for_fair_bbn, "adult"),
        ("BANK", preprocess_bank_for_fair_bbn, "bank"),
        ("LAWSCHOOL", preprocess_lawschool_for_fair_bbn, "lawschool"),
        ("DIABETES HOSPITAL", preprocess_diabetes_hospital_for_fair_bbn, "hospital"),
    ]

    for name, preprocess_func, dataset_name in datasets:
        print(f"\n{'=' * 60}\n▶ PROCESSING {name} DATASET\n{'=' * 60}")
        data = preprocess_func()
        results = train_fair_bbn_adapter(data, dataset_name)
        print(f"{name} dataset completed successfully")

    print("\n" + "=" * 60)
    print("FAIR BBN SYSTEM EXECUTION COMPLETE")
    print("=" * 60)

Starting Fair BBN System on Kaggle

▶ PROCESSING GERMAN CREDIT DATASET
Training baseline MLP...
Baseline MLP Accuracy: 0.8571
Training fair BBN...
Training adapter with adversarial + BBN fairness...
Epoch   0 | Adv Loss: 0.6125 | Adapter Loss: 0.3911
Epoch  10 | Adv Loss: 0.4823 | Adapter Loss: 0.4565
Epoch  20 | Adv Loss: 0.3874 | Adapter Loss: 0.5010
Epoch  30 | Adv Loss: 0.3206 | Adapter Loss: 0.5333
Epoch  40 | Adv Loss: 0.3042 | Adapter Loss: 0.5415
Epoch  50 | Adv Loss: 0.2974 | Adapter Loss: 0.5446
Epoch  59 | Adv Loss: 0.2842 | Adapter Loss: 0.5510

BASELINE MLP RESULTS:
 Accuracy: 0.8571
 age_dp: 0.0940
 age_eo: 0.0731
 sex_dp: 0.0461
 sex_eo: 0.0945
 foreign_dp: 0.4361
 foreign_eo: 0.2188

BBN + Adapter RESULTS:
 Accuracy: 0.5000
 age_dp: 0.0000
 age_eo: 0.0000
 sex_dp: 0.0000
 sex_eo: 0.0000
 foreign_dp: 0.0000
 foreign_eo: 0.0000
GERMAN CREDIT dataset completed successfully

▶ PROCESSING COMPAS DATASET


/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)


Training baseline MLP...
Baseline MLP Accuracy: 0.6372
Training fair BBN...
Training adapter with adversarial + BBN fairness...
Epoch   0 | Adv Loss: 1.2671 | Adapter Loss: 0.0716
Epoch  10 | Adv Loss: 0.8446 | Adapter Loss: 0.2710
Epoch  20 | Adv Loss: 0.8083 | Adapter Loss: 0.2859
Epoch  30 | Adv Loss: 0.8049 | Adapter Loss: 0.2868
Epoch  40 | Adv Loss: 0.8009 | Adapter Loss: 0.2882
Epoch  50 | Adv Loss: 0.8063 | Adapter Loss: 0.2856
Epoch  59 | Adv Loss: 0.8036 | Adapter Loss: 0.2866

BASELINE MLP RESULTS:
 Accuracy: 0.6372
 race_dp: 0.5538
 race_eo: 0.6558
 sex_dp: 0.1319
 sex_eo: 0.1136

BBN + Adapter RESULTS:
 Accuracy: 0.5449
 race_dp: 0.0000
 race_eo: 0.0000
 sex_dp: 0.0000
 sex_eo: 0.0000
COMPAS dataset completed successfully

▶ PROCESSING ADULT INCOME DATASET
Training baseline MLP...


/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:686: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


Baseline MLP Accuracy: 0.8746
Training fair BBN...
Training adapter with adversarial + BBN fairness...
Epoch   0 | Adv Loss: 0.9060 | Adapter Loss: 0.2430
Epoch  10 | Adv Loss: 0.5357 | Adapter Loss: 0.4253
Epoch  20 | Adv Loss: 0.5355 | Adapter Loss: 0.4254
Epoch  30 | Adv Loss: 0.5354 | Adapter Loss: 0.4255
Epoch  40 | Adv Loss: 0.5357 | Adapter Loss: 0.4253
Epoch  50 | Adv Loss: 0.5356 | Adapter Loss: 0.4254
Epoch  59 | Adv Loss: 0.5356 | Adapter Loss: 0.4254

BASELINE MLP RESULTS:
 Accuracy: 0.8746
 race_dp: 0.3269
 race_eo: 0.1584
 sex_dp: 0.3241
 sex_eo: 0.1649

BBN + Adapter RESULTS:
 Accuracy: 0.5000
 race_dp: 0.0000
 race_eo: 0.0000
 sex_dp: 0.0000
 sex_eo: 0.0000
ADULT INCOME dataset completed successfully

▶ PROCESSING BANK DATASET
Training baseline MLP...


/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:686: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


Baseline MLP Accuracy: 0.9290
Training fair BBN...
Training adapter with adversarial + BBN fairness...
Epoch   0 | Adv Loss: 1.7198 | Adapter Loss: -0.1652
Epoch  10 | Adv Loss: 1.5221 | Adapter Loss: -0.0679
Epoch  20 | Adv Loss: 1.5220 | Adapter Loss: -0.0678
Epoch  30 | Adv Loss: 1.5221 | Adapter Loss: -0.0679
Epoch  40 | Adv Loss: 1.5221 | Adapter Loss: -0.0679
Epoch  50 | Adv Loss: 1.5222 | Adapter Loss: -0.0679
Epoch  59 | Adv Loss: 1.5221 | Adapter Loss: -0.0679

BASELINE MLP RESULTS:
 Accuracy: 0.9290
 marital_dp: 0.1137
 marital_eo: 0.0642
 job_dp: 0.4521
 job_eo: 0.3489

BBN + Adapter RESULTS:
 Accuracy: 0.5002
 marital_dp: 0.0000
 marital_eo: 0.0000
 job_dp: 0.0000
 job_eo: 0.0000
BANK dataset completed successfully

▶ PROCESSING LAWSCHOOL DATASET
Final dataset — X_train_nn: (32916, 23), BBN_train: (32916, 42)
Training baseline MLP...
Baseline MLP Accuracy: 1.0000
Training fair BBN...
Training adapter with adversarial + BBN fairness...
Epoch   0 | Adv Loss: 1.2323 | Adapter 

/tmp/ipykernel_37/3641517833.py:24: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['age'] = df['age'].replace(age_map).astype(float)


Training baseline MLP...
Baseline MLP Accuracy: 0.5979
Training fair BBN...
Training adapter with adversarial + BBN fairness...
Epoch   0 | Adv Loss: 1.0774 | Adapter Loss: nan
Epoch  10 | Adv Loss: 0.7644 | Adapter Loss: nan
Epoch  20 | Adv Loss: 0.7644 | Adapter Loss: nan
Epoch  30 | Adv Loss: 0.7645 | Adapter Loss: nan
Epoch  40 | Adv Loss: 0.7645 | Adapter Loss: nan
Epoch  50 | Adv Loss: 0.7645 | Adapter Loss: nan
Epoch  59 | Adv Loss: 0.7644 | Adapter Loss: nan

BASELINE MLP RESULTS:
 Accuracy: 0.5979
 race_dp: 0.1837
 race_eo: 0.1715
 sex_dp: 0.0158
 sex_eo: 0.0345

BBN + Adapter RESULTS:
 Accuracy: 0.5580
 race_dp: 0.0000
 race_eo: 0.0000
 sex_dp: 0.0000
 sex_eo: 0.0000
DIABETES HOSPITAL dataset completed successfully

FAIR BBN SYSTEM EXECUTION COMPLETE
